# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [114]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [115]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)

nltk.download('punkt_tab', quiet=True) # Daba un error con versiones de nltk asi que tuve que añadirle el _tab
# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
def protect_acronym_dots(text):
    # Replace dots with <DOT> for the nltk
    return re.sub(r'\b([A-Z])\.', r'\1<DOT>', text)

def restore_acronym_dots(text):
    # Restore dots in acronyms
    return re.sub(r'<DOT>', '.', text)

# Protect acronyms
print("Protected text:") 
protected_text = protect_acronym_dots(text) # Protejo el texto para que nltk no lo divida mal
print(protected_text)

# TODO: apply sent_tokenize
print("\n Tokenize:")
sentences = sent_tokenize(protected_text) # aplico el sent_tokenize
print(sentences)

print("\n Restauradas:")
final = [restore_acronym_dots(sentence) for sentence in sentences] # Restauro el texto
print(final)

# print(sentences)


Protected text:
In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U<DOT>P<DOT>C<DOT> and U<DOT>N<DOT>E<DOT>S<DOT>C<DOT>O<DOT> A report valued the project at $3.2 billion.

 Tokenize:
['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U<DOT>P<DOT>C<DOT> and U<DOT>N<DOT>E<DOT>S<DOT>C<DOT>O<DOT> A report valued the project at $3.2 billion.']

 Restauradas:
['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.']


## Q2

In [116]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
#
# Return: text_norm

def normalize_text(text):
    # Quitar puntos solo en acronimos 
    text = re.sub(r'\b[A-Z\.][A-Z\.]+', lambda m: re.sub(r'\.', '', m.group()), text)
    # Quitar el simbolo del dolar
    text = re.sub(r'\$', '', text)
    return text


text_norm = normalize_text(text)


print(text_norm)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from UPC and UNESCO A report valued the project at 3.2 billion.


## Q3

In [ ]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case

text_case = None

# print(text_case)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from UPC and UNESCO A report valued the project at 3.2 billion.


## Q4

In [118]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

tokens = word_tokenize(text) 
print(tokens)


['In', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam', 'Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '1.86m', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'U.P.C', '.', 'and', 'U.N.E.S.C.O', '.', 'A', 'report', 'valued', 'the', 'project', 'at', '$', '3.2', 'billion', '.']


## Q5

In [119]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop

def remove_stopwords(tokens):
    # Create a list of English stopwords
    stop_words = set(stopwords.words('english'))
    
    # keep only tokens that are not stopwords
    tokens_nostop = [token for token in tokens 
                    if token.lower() not in stop_words]
    
    return tokens_nostop

tokens_nostop = remove_stopwords(tokens)
print(tokens_nostop)

['mid-February', '2026', ',', 'CEO', 'OpenAI', ',', 'Sam', 'Altman', ',', 'visited', 'Barcelona', '.', '1.86m', 'tall', 'met', 'researchers', 'U.P.C', '.', 'U.N.E.S.C.O', '.', 'report', 'valued', 'project', '$', '3.2', 'billion', '.']


## Q6

In [128]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

def get_bigrams(tokens):
    bigrams = []
    for i in range(len(tokens)-1): # -1 beacouse of the index errror I had
        bigrams.append((tokens[i], tokens[i+1])) # create biagrams by pairing each token with the next one
    return bigrams

bigrams = get_bigrams(tokens)
print(bigrams)


[('In', 'mid-February'), ('mid-February', '2026'), ('2026', ','), (',', 'the'), ('the', 'CEO'), ('CEO', 'of'), ('of', 'OpenAI'), ('OpenAI', ','), (',', 'Sam'), ('Sam', 'Altman'), ('Altman', ','), (',', 'visited'), ('visited', 'Barcelona'), ('Barcelona', '.'), ('.', 'He'), ('He', 'is'), ('is', '1.86m'), ('1.86m', 'tall'), ('tall', 'and'), ('and', 'met'), ('met', 'with'), ('with', 'researchers'), ('researchers', 'from'), ('from', 'U.P.C'), ('U.P.C', '.'), ('.', 'and'), ('and', 'U.N.E.S.C.O'), ('U.N.E.S.C.O', '.'), ('.', 'A'), ('A', 'report'), ('report', 'valued'), ('valued', 'the'), ('the', 'project'), ('project', 'at'), ('at', '$'), ('$', '3.2'), ('3.2', 'billion'), ('billion', '.')]


## Q7

In [ ]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = Counter(bigrams)
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


ZeroDivisionError: division by zero

## Q8

In [122]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [123]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = 44
FP = 3
FN = 10
TN = 40

accuracy = (TP + TN) / (TP + FP + FN + TN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * (precision * recall) / (precision + recall)

print(accuracy, precision, recall, f1)


0.865979381443299 0.9361702127659575 0.8148148148148148 0.8712871287128713
